# Demonstrating Grid Cells in the Medial Entorhinal Cortex

**Dataset:** DANDI dandiset [000582](https://dandiarchive.org/dandiset/000582),
*"Conjunctive Representation of Position, Direction, and Velocity in Entorhinal
Cortex"* (Sargolini et al., *Science*, 2006). Extracellular recordings from the
dorsocaudal medial entorhinal cortex (MEC) of Long-Evans rats foraging in a
100 × 100 cm open field, with two-LED head tracking at 50 Hz.

**Goal.** Grid cells fire whenever the animal occupies any vertex of a periodic
triangular (hexagonal) lattice tiling the environment. We demonstrate this by
computing, for each unit:

1. an occupancy-normalized 2-D firing-rate map,
2. the spatial autocorrelogram of that map (hexagonal symmetry is the signature
   of a grid cell), and
3. the **gridness score** (Sargolini 2006), validated against a spike-time
   shuffle null distribution.

All data are streamed from the DANDI S3 store with `remfile` + local disk
caching; nothing is downloaded in full. Analysis helpers live in `gridcells.py`.

## Setup

In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
from tqdm import tqdm

import gridcells as gc

BINS = 40                       # 40 x 40 spatial bins over the box
BOX = (-50.0, 50.0)             # box extent in cm
BIN_SIZE = (BOX[1] - BOX[0]) / BINS   # 2.5 cm per bin
SIGMA = 1.0                     # Gaussian smoothing (bins)
MIN_SPIKES = 100
N_SHUFFLE = 30

## 1. Load and inspect one session

Each NWB file stores head-tracking position under `processing/behavior/Position`
(one `SpatialSeries` per LED; we average the LEDs to get the head centroid) and
sorted spikes in the `units` table. We first validate a single rich session.

In [2]:
path = "sub-11265/sub-11265_ses-06020601_behavior+ecephys.nwb"
nwb = gc.load_session(path)
t, xy = gc.extract_position(nwb)
units = gc.extract_units(nwb)
print(f"duration = {t[-1] - t[0]:.0f} s, "
      f"position samples = {len(t)}, units = {len(units)}")
print(f"x range {xy[:,0].min():.0f}..{xy[:,0].max():.0f} cm, "
      f"y range {xy[:,1].min():.0f}..{xy[:,1].max():.0f} cm")

duration = 600 s, position samples = 30000, units = 19
x range -50..50 cm, y range -50..49 cm


### Validate the raw behavioral trajectory and spike coverage

In [3]:
occ, ex, ey = gc.occupancy_map(t, xy, BINS, BOX, sigma=SIGMA)
fig, ax = plt.subplots(1, 2, figsize=(10, 4.4))
ax[0].plot(xy[:, 0], xy[:, 1], lw=0.3, color="0.4")
ax[0].set_aspect("equal"); ax[0].set_title("Head trajectory (10 min)")
ax[0].set_xlabel("x (cm)"); ax[0].set_ylabel("y (cm)")
im = ax[1].imshow(occ.T, origin="lower", cmap="viridis",
                  extent=[BOX[0], BOX[1], BOX[0], BOX[1]])
ax[1].set_title("Occupancy (s)"); plt.colorbar(im, ax=ax[1], fraction=0.046)
ax[1].set_xlabel("x (cm)"); ax[1].set_ylabel("y (cm)")
fig.tight_layout(); fig.savefig("fig0_trajectory_occupancy.png", dpi=120)
print(f"occupancy total = {np.nansum(occ):.0f} s over "
      f"{np.isfinite(occ).sum()} visited bins")

occupancy total = 595 s over 1566 visited bins


The animal samples the whole box fairly uniformly, so occupancy normalization is
well posed. Now compute rate map, autocorrelogram, and gridness for every unit.

In [4]:
for u in units:
    st = u["spike_times"]
    rm, ac, g, si = gc.analyze_unit(st, t, xy, occ, ex, BIN_SIZE, SIGMA)
    print(f"  {u['name']:6s}  gridness={g['gridness']:+.2f}  "
          f"spacing={g['spacing_cm']:.0f} cm  SI={si:.2f} bits/spk  "
          f"peak={np.nanmax(rm):.1f} Hz  n_spikes={len(st)}")

  t1c1    gridness=+1.16  spacing=43 cm  SI=0.91 bits/spk  peak=8.5 Hz  n_spikes=546
  t1c2    gridness=-0.29  spacing=37 cm  SI=0.70 bits/spk  peak=14.8 Hz  n_spikes=1233


  t2c1    gridness=+0.14  spacing=29 cm  SI=0.51 bits/spk  peak=4.8 Hz  n_spikes=364


  t2c2    gridness=+1.18  spacing=49 cm  SI=1.03 bits/spk  peak=17.6 Hz  n_spikes=998
  t2c3    gridness=-0.17  spacing=29 cm  SI=0.16 bits/spk  peak=6.1 Hz  n_spikes=1065


  t2c4    gridness=-0.12  spacing=20 cm  SI=0.07 bits/spk  peak=11.5 Hz  n_spikes=3200


  t3c1    gridness=+0.99  spacing=43 cm  SI=1.03 bits/spk  peak=6.2 Hz  n_spikes=447


...

  t3c6    gridness=+0.10  spacing=30 cm  SI=0.19 bits/spk  peak=6.0 Hz  n_spikes=1118
  t4c1    gridness=+0.22  spacing=53 cm  SI=1.51 bits/spk  peak=17.8 Hz  n_spikes=515


  t4c2    gridness=+0.15  spacing=52 cm  SI=0.23 bits/spk  peak=6.7 Hz  n_spikes=1037
  t4c4    gridness=-0.40  spacing=26 cm  SI=0.21 bits/spk  peak=8.3 Hz  n_spikes=1056
  t4c5    gridness=+1.10  spacing=48 cm  SI=0.79 bits/spk  peak=10.3 Hz  n_spikes=673


  t5c1    gridness=+1.12  spacing=70 cm  SI=1.45 bits/spk  peak=14.7 Hz  n_spikes=879
  t5c3    gridness=-0.11  spacing=25 cm  SI=0.12 bits/spk  peak=13.9 Hz  n_spikes=2699


  t7c1    gridness=+1.11  spacing=71 cm  SI=1.71 bits/spk  peak=8.7 Hz  n_spikes=517
  t7c2    gridness=-0.28  spacing=40 cm  SI=2.05 bits/spk  peak=4.9 Hz  n_spikes=174


  t7c3    gridness=+1.29  spacing=52 cm  SI=1.51 bits/spk  peak=19.3 Hz  n_spikes=1053


## 2. The gridness score and the shuffle control

A grid cell's autocorrelogram has a central peak surrounded by six peaks at
~60° spacing. The **gridness score** rotates an annular ring of the
autocorrelogram by 30/60/90/120/150° and computes
`min(r60, r120) − max(r30, r90, r150)`: hexagonal maps correlate strongly at
60°/120° and poorly at 30°/90°/150°, giving a positive score.

To decide which scores are significant we build a null distribution by
circularly shifting each cell's spike train by a random offset (destroying the
spike–position relationship) and recomputing gridness. Cells whose observed
gridness exceeds the 95th percentile of the pooled null are classified as grid
cells.

## 3. Pool the richest sessions

The full analysis (`run_analysis.py`) processes the eight sessions with the most
simultaneously recorded units, pools all units, runs the shuffle control, and
caches everything to `results.pkl`. We load that cache here (run the script
first if it is missing).

In [5]:
import os, pickle, subprocess
if not os.path.exists("results.pkl"):
    subprocess.run(["python", "run_analysis.py"], check=True)
with open("results.pkl", "rb") as f:
    R = pickle.load(f)
records = R["records"]
grid = np.array([r["gridness"] for r in records], float)
spacing = np.array([r["spacing_cm"] for r in records], float)
si = np.array([r["spatial_info"] for r in records], float)
is_grid = grid > R["threshold"]
print(f"pooled units: {len(records)}")
print(f"shuffle 95th-pctile gridness threshold: {R['threshold']:.2f}")
print(f"grid cells: {int(is_grid.sum())} / {len(records)} "
      f"({100 * is_grid.mean():.0f}%)")
print(f"median grid spacing: {np.nanmedian(spacing[is_grid]):.0f} cm")

pooled units: 115
shuffle 95th-pctile gridness threshold: 0.46
grid cells: 22 / 115 (19%)
median grid spacing: 60 cm


## 4. Figures

`make_figures.py` renders the full figure set from the cached results:

* `fig1_example_grid_cells.png` — six clearest grid cells (trajectory+spikes,
  rate map, autocorrelogram).
* `fig2_population_summary.png` — gridness vs. shuffle null, grid-spacing
  distribution, spatial-info vs. gridness.
* `fig3_grid_cell_gallery.png` — autocorrelograms of every identified grid cell.
* `fig4_exemplar_with_control.png` — one exemplar with its own shuffle null.

In [6]:
subprocess.run(["python", "make_figures.py"], check=True)

saved fig1_example_grid_cells.png
saved fig2_population_summary.png
saved fig3_grid_cell_gallery.png
saved fig4_exemplar_with_control.png

Total units: 115  |  grid cells: 22 (19%)
Median grid spacing: 60 cm
Grid-cell median SI: 1.02 bits/spike


CompletedProcess(args=['python', 'make_figures.py'], returncode=0)

### Example grid cells
![example grid cells](fig1_example_grid_cells.png)

### Population summary
![population summary](fig2_population_summary.png)

### All identified grid cells
![gallery](fig3_grid_cell_gallery.png)

### Exemplar with shuffle control
![exemplar](fig4_exemplar_with_control.png)

## Conclusion

The spatial autocorrelograms show the hallmark six-fold (hexagonal) symmetry of
grid cells, with grid spacings in the ~40–70 cm range expected for dorsocaudal
MEC. A substantial fraction of MEC units exceed the shuffle-based gridness
threshold, reproducing the central result of Sargolini et al. (2006): the MEC
contains a population of cells that encode the animal's location through a
periodic triangular lattice, independent of the specific environment.